In [ ]:
import os, anthropic
from pathlib import Path
from dotenv import load_dotenv

def find_project_root(start):
    for p in [start, *start.parents]:
        if (p / "data" / "metrics_159d.csv").exists():
            return p

ROOT = find_project_root(Path.cwd())
load_dotenv(ROOT / ".env")

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print("Ready")

In [ ]:
design_doc_prompt = """
You are a senior product manager writing a design doc for a take-home assignment at a D2C analytics SaaS company. Write a professional, specific, technically grounded design doc. No filler. No generic statements. Every claim must be concrete.

USE THESE REAL FINDINGS FROM OUR EDA AND PROTOTYPE:

Dataset facts:
- 160 days of data, Sep 2025 - Feb 2026
- Sources: meta_ads, google_ads, shopify
- Median daily revenue: INR 22.6L
- Revenue range: INR 1.15L (min, likely incomplete) to INR 56.5L (max)
- Best day: Thursday. Worst day: Sunday
- Anomaly days (|z|>2): 4 days — Oct 8, Dec 10, Jan 1, Feb 7
- Ad spend correlation with revenue: Meta r=0.12, Google r=0.15 (near-zero)
- ~91% of Shopify orders unattributed at channel level

Ranker design decisions made:
- Z-score uses 28-day rolling window with DoW adjustment (compare Monday to recent Mondays)
- WoW delta capped at 500% for ratio metrics (ROAS, CTR, CPM, CPC) to prevent near-zero spend artifacts
- Business weights: revenue=1.0, orders=0.9, roas=0.8, aov=0.75, spend=0.7
- Deduplication: shopify total and unattributed channel slice excluded to prevent double-counting
- Last-day data quality flag for incomplete export days
- Scoring: final_score = (|z|/3×0.4 + |wow|/100×0.4 + |mom|/100×0.2) × business_weight
- Sustained signals (3+ days same direction) boosted by 10% per day in weekly ranker

Real signals found in the data:
- Dec 10: new customer revenue +416.5% WoW, new customer AOV +190.5% WoW — likely sale event
- Jan 29: direct channel revenue +370.7% MoM, AOV INR 33,679 vs baseline INR 12,312 — high-intent cohort
- Week of Feb 1: search_google_002 ROAS hit exactly 0.000 — campaign spent with zero attributed orders
- Week of Feb 7: new customer orders +243.8% WoW while overall Feb 7 revenue crashed 92% (incomplete data)

LLM design decisions made:
- Model: claude-sonnet-4-6
- All numbers passed as pre-computed fact sentences — LLM cannot invent values
- System prompt enforces: hedged causation language, capped-value disclosure, data quality flags
- Two output formats: Daily Digest (5 sections) and Weekly Report (5 sections)
- Two customer profiles: growth-stage (ROAS/acquisition focus) vs scale-stage (AOV/retention focus)

Write the design doc in markdown with these exact sections:

# Daily Digest & Weekly Report — Product & System Design

## 1. Users and Job-to-be-Done

### Daily Digest
Who reads it, when, and what decision does it enable? Be specific about the user's morning workflow.

### Weekly Report  
Different job. Different reader mode. Explain the distinction concretely.

## 2. What Makes a Metric Movement "Important"

### Statistical scoring
Explain z-score, DoW adjustment, WoW/MoM deltas with the actual formula used.
Use the real threshold values and weights from above.

### Business weighting
List the actual weights. Explain why revenue=1.0 and impressions=0.2.

### The double-counting problem
Explain the shopify/unattributed dedup issue discovered in EDA and how it was solved.

### The ratio metric artifact problem  
Explain the ROAS spike cap. Use the real example: shopping_google_003 showed 4653% WoW before capping.

## 3. Personalization

### Cold start
What does a new customer get? What defaults are used?

### Warm customer (2-8 weeks)
What signals start to accumulate? How do they shift the output?

### Mature customer (2+ months)
What does the system know? How does the output differ from cold start?

### The two profiles built
Describe PROFILE_GROWTH and PROFILE_SCALE with their actual parameters.

## 4. System Architecture

### Batch layer (nightly)
What runs on a schedule? What does it produce?

### Online layer (per-report)
What happens at report generation time? Latency budget?

### Deterministic vs generative boundary
Where exactly does deterministic end and LLM begin? Why is this boundary placed here?

### Data flow diagram (text)
ASCII or markdown table showing: raw CSV → ranker → ranked_findings.json → report_generator → .md reports

## 5. Top 3 Failure Modes and Mitigations

### Failure 1: Hallucinated causation
Describe the risk. Explain the mitigation (pre-computed fact sentences, system prompt rules, hedged language enforcement).

### Failure 2: Alert fatigue
Describe the risk (too many alerts every day). Explain the mitigation (business weights, DoW adjustment, top-N cap, sustained signal boosting).

### Failure 3: Last-day incomplete data
Describe the real example found: Feb 7 showed revenue -92% WoW. Explain the mitigation (data quality flag, sort-to-bottom logic).

## 6. How We Know It's Working

List 5 specific product metrics (not model metrics) with their measurement method:
- Open rate
- Follow-up question rate  
- "Not relevant" click rate
- 7-day retention after first digest
- Alert-to-action conversion rate

For each: what is it, how is it measured, what threshold indicates success?

Write in clear, direct prose. No bullet spam. Use tables where they aid clarity. 4-5 pages equivalent. This is being read by a technical CEO and engineering panel.
"""

response = client.messages.create(
    model      = "claude-sonnet-4-6",
    max_tokens = 4000,
    messages   = [{"role": "user", "content": design_doc_prompt}]
)

design_doc = response.content[0].text
print(design_doc[:500])
print("...")
print(f"\nTotal length: {len(design_doc):,} chars")

In [ ]:
path = ROOT / "design_doc.md"
with open(path, "w", encoding="utf-8") as f:
    f.write(design_doc)
print(f"Saved: {path}")
print(f"Lines: {len(design_doc.splitlines())}")

In [ ]:
tradeoffs_prompt = """
Write a concise, honest 1-page "Trade-offs & What's Next" section for a Senior AI Product Engineer take-home assignment submission.

Context of what was built:
- Deterministic ranker: z-score (28d rolling, DoW-adjusted) + WoW/MoW delta + business weights
- LLM layer: claude-sonnet-4-6, grounded fact sentences, hedged causation rules
- Two customer profiles: growth-stage (acquisition focus) vs scale-stage (retention focus)
- 5 sample reports generated from 160 days of real anonymized D2C data
- Built in: Python 3.9, pandas, anthropic SDK, Jupyter

What was deliberately cut and why:
1. Personalization ML model — no click-stream data available; rule-based profiles used instead
2. GA4 sessions, bounce rate, conversion funnel — not in dataset
3. Email delivery infrastructure — out of scope for prototype
4. A/B testing framework for report variants — needs real engagement data first
5. Hourly granularity — dataset is daily only
6. SKU/product-level breakdowns — not in dataset
7. Real-time alerting — batch is sufficient for daily digest use case

What to build next (in priority order):
1. Engagement feedback loop — "Was this useful?" thumbs, click tracking on findings
2. Metric preference learning — weight adjustments from engagement signals
3. Calendar event awareness — suppress Jan 1 anomaly, flag known sale days
4. Cross-tenant pattern library — "brands like yours typically see X on Thursdays"
5. Explanation confidence scoring — rate each causal hypothesis by data support

What to learn from real customers first:
- Do they read daily or skip to weekly?
- Which finding types drive follow-up actions vs ignored?
- What's the tolerance for false positives vs false negatives?
- Do growth-stage and scale-stage brands actually behave differently?

Scale failure points:
- 100 customers: LLM latency (sequential calls ~3s each = 5 min batch), report differentiation starts looking similar
- 10,000 customers: LLM cost (~$0.15/report × 2 reports × 10,000 = $3,000/day), prompt caching needed, tenant data isolation, nightly batch SLA risk

Write in direct markdown prose. Honest about limitations. Shows engineering maturity.
Use these exact sections:

# Trade-offs & What's Next

## What I cut and why

## What I'd build next

## What I need to learn from real customers first

## Where this breaks at scale
"""

response = client.messages.create(
    model      = "claude-sonnet-4-6",
    max_tokens = 1500,
    messages   = [{"role": "user", "content": tradeoffs_prompt}]
)

tradeoffs = response.content[0].text
print(tradeoffs)

In [ ]:
path = ROOT / "tradeoffs.md"
with open(path, "w", encoding="utf-8") as f:
    f.write(tradeoffs)
print(f"Saved: {path}")

In [ ]:
print("=" * 60)
print("SUBMISSION INVENTORY")
print("=" * 60)

# Design doc
dd = ROOT / "design_doc.md"
print(f"\n1. Design doc ({dd.stat().st_size:,} bytes)")
print(f"   {dd}")

# Tradeoffs
td = ROOT / "tradeoffs.md"
print(f"\n2. Tradeoffs page ({td.stat().st_size:,} bytes)")
print(f"   {td}")

# Sample outputs
reports = sorted((ROOT / "outputs" / "reports").glob("*.md"))
print(f"\n3. Sample outputs ({len(reports)} reports)")
for r in reports:
    print(f"   {r.name:45s} {r.stat().st_size:,} bytes")

# Source code
src_files = list((ROOT / "src").glob("*.py"))
print(f"\n4. Source code ({len(src_files)} files)")
for s in src_files:
    print(f"   {s.name:45s} {s.stat().st_size:,} bytes")

# Notebooks
nbs = list((ROOT / "notebooks").glob("*.ipynb"))
print(f"\n5. Notebooks ({len(nbs)} files)")
for n in nbs:
    print(f"   {n.name}")

print("\n" + "=" * 60)
print("README note to add:")
print("""
## How I used AI tools
- Claude (claude-sonnet-4-6) used for:
  - Drafting and iterating the design doc
  - Generating the tradeoffs analysis
  - LLM layer in report_generator.py
- All ranking logic (ranker.py) is deterministic Python — no LLM
- All numbers in generated reports are pre-computed facts — LLM cannot invent values
- EDA and data insights discovered through manual notebook exploration
""")